<h1>Models Inference<h1>

In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, confusion_matrix
from typing import Dict, List, Tuple
import warnings
warnings.filterwarnings('ignore')
print("✅ Imports successful!")


✅ Imports successful!


In [ ]:
# TCGA trained models base path
TCGA_MODELS_BASE = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\slide_classification\TCGA_Result\5-Caption_based_aggregation"

# PAIP data path
PAIP_DATA_BASE = r"D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\Caption_Based_Clustering_FiveCrop"

# Ground truth labels
GROUND_TRUTH_PATH = r"D:\Aamir Gulzar\KSA_project2\paip_data\labels\paip_78_labels.csv"

# Output path for results
OUTPUT_PATH = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\slide_classification\Inference_Results"
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Model configurations
FEATURE_MODELS = ['Conch1_5', 'H-Optimus-1', 'UNI2']
PAIP_FEATURE_DIRS = {
    'Conch1_5': 'conch_CC_fivecrop',
    'H-Optimus-1': 'hoptimus_CC_fivecrop',
    'UNI2': 'uni2_CC_fivecrop'
}
CLASSIFIER_TYPES = {
    'logistic_regression': 'fold{}_logistic_regression.pkl',
    'ann': 'fold{}_ann.pth',
    'knn': 'fold{}_knn.pkl',
    'protonet': 'fold{}_protonet.pkl',
    'random_forest': 'fold{}_random_forest.pkl'
}
TASK = 'MSIH'
NUM_FOLDS = 3

print("✅ Configuration set!")
print(f"TCGA Models: {TCGA_MODELS_BASE}")
print(f"PAIP Data: {PAIP_DATA_BASE}")
print(f"Output: {OUTPUT_PATH}")


✅ Configuration set!
TCGA Models: D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\slide_classification\TCGA_Result\5-Caption_based_aggregation
PAIP Data: D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\Caption_Based_Clustering_FiveCrop
Output: D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\slide_classification\Inference_Results


In [ ]:

def load_paip_features(feature_dir: str) -> Tuple[Dict[str, torch.Tensor], List[str]]:
    """Load all PAIP feature files from a directory"""
    features_dict = {}
    wsi_ids = []
    
    print(f"Loading features from: {feature_dir}")
    files = [f for f in os.listdir(feature_dir) if f.endswith('.pt')]
    print(f"Found {len(files)} feature files")
    
    for file in files:
        wsi_id = os.path.splitext(file)[0]
        file_path = os.path.join(feature_dir, file)
        
        try:
            features = torch.load(file_path)
            if features.is_cuda:
                features = features.cpu()
            
            # Flatten if needed
            if features.dim() > 1:
                features = features.flatten()
            
            features_dict[wsi_id] = features
            wsi_ids.append(wsi_id)
        except Exception as e:
            print(f"Error loading {file}: {e}")
    
    return features_dict, wsi_ids



In [ ]:
"""
TCGA Models Inference on PAIP Data
This script loads trained models from TCGA dataset (fold3 only) and runs inference on PAIP data
"""

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import (
    accuracy_score, 
    balanced_accuracy_score, 
    f1_score, 
    roc_auc_score, 
    confusion_matrix
)
from typing import Dict, List, Tuple
import warnings
warnings.filterwarnings('ignore')

# ============================================
# CONFIGURATION
# ============================================
TCGA_MODELS_BASE = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\slide_classification\TCGA_Result\5-Caption_based_aggregation"
PAIP_DATA_BASE = r"D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\Caption_Based_Clustering_FiveCrop"
GROUND_TRUTH_PATH = r"D:\Aamir Gulzar\KSA_project2\paip_data\labels\paip_78_labels.csv"
OUTPUT_PATH = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\slide_classification\Inference_Results"
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Model configurations
FEATURE_MODELS = ['Conch1_5', 'H-Optimus-1', 'UNI2']
PAIP_FEATURE_DIRS = {
    'Conch1_5': 'conch_CC_fivecrop',
    'H-Optimus-1': 'hoptimus_CC_fivecrop',
    'UNI2': 'uni2_CC_fivecrop'
}
CLASSIFIER_TYPES = {
    'logistic_regression': 'fold3_logistic_regression.pkl',
    'ann': 'fold3_trained_ann_model_10752.pth',
    'knn': 'fold3_knn_model.pkl',
    'protonet': 'fold3_protonet_model.pkl'
}
TASK = 'MSIH'
FOLD_TO_USE = 3  # Only using fold3


# ============================================
# HELPER FUNCTIONS
# ============================================

def load_paip_features(feature_dir: str) -> Tuple[Dict[str, torch.Tensor], List[str]]:
    """Load all PAIP feature files from a directory"""
    features_dict = {}
    wsi_ids = []
    
    print(f"Loading features from: {feature_dir}")
    files = [f for f in os.listdir(feature_dir) if f.endswith('.pt')]
    print(f"Found {len(files)} feature files")
    
    for file in files:
        wsi_id = os.path.splitext(file)[0]
        file_path = os.path.join(feature_dir, file)
        
        try:
            features = torch.load(file_path)
            if features.is_cuda:
                features = features.cpu()
            
            # Flatten if needed
            if features.dim() > 1:
                features = features.flatten()
            
            features_dict[wsi_id] = features
            wsi_ids.append(wsi_id)
        except Exception as e:
            print(f"Error loading {file}: {e}")
    
    return features_dict, wsi_ids


def load_ground_truth(csv_path: str) -> pd.DataFrame:
    """Load ground truth labels from CSV"""
    df = pd.read_csv(csv_path)
    print(f"Loaded ground truth with {len(df)} samples")
    print(f"Columns: {df.columns.tolist()}")
    return df


def get_label_from_wsi_id(wsi_id: str, ground_truth_df: pd.DataFrame) -> int:
    """Extract label from ground truth dataframe"""
    if 'WSI_Id' in ground_truth_df.columns:
        match = ground_truth_df[ground_truth_df['WSI_Id'] == wsi_id]
        if not match.empty:
            if 'label' in ground_truth_df.columns:
                label_val = match['label'].values[0]
            elif 'label_desc' in ground_truth_df.columns:
                label_val = match['label_desc'].values[0]
            else:
                raise ValueError("No label column found")
            
            # Convert to binary
            if isinstance(label_val, str):
                return 0 if label_val.lower() == 'nonmsih' else 1
            else:
                return int(label_val)
    
    return None


def calculate_metrics(y_true: np.ndarray, y_pred: np.ndarray, y_proba: np.ndarray = None) -> Dict:
    """Calculate classification metrics"""
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'f1_macro': f1_score(y_true, y_pred, average='macro'),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted'),
        'confusion_matrix': confusion_matrix(y_true, y_pred).tolist()
    }
    
    if y_proba is not None and len(np.unique(y_true)) == 2:
        try:
            if y_proba.ndim > 1:
                y_proba = y_proba[:, 1]
            metrics['auroc'] = roc_auc_score(y_true, y_proba)
        except:
            metrics['auroc'] = None
    
    return metrics


# ============================================
# MODEL LOADING FUNCTIONS
# ============================================
class ANNClassifier(nn.Module):
    """ANN Classifier architecture"""
    def __init__(self, input_dim, hidden_dim=256, hidden_dim2=128, num_classes=2):
        super(ANNClassifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.3)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim2)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.3)
        self.fc3 = nn.Linear(hidden_dim2, num_classes)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        x = self.fc3(x)
        return x


def load_sklearn_model(model_path: str):
    """Load sklearn-based models"""
    return joblib.load(model_path)


def load_ann_model(model_path: str, input_dim: int, hidden_dim: int = 256, hidden_dim2: int = 128):
    """Load PyTorch ANN model"""
    model = ANNClassifier(input_dim, hidden_dim, hidden_dim2)
    model.load_state_dict(torch.load(model_path, map_location='cpu'))
    model.eval()
    return model


def predict_sklearn(model, features: torch.Tensor):
    """Make predictions with sklearn models"""
    X = features.numpy() if isinstance(features, torch.Tensor) else features
    if X.ndim == 1:
        X = X.reshape(1, -1)
    
    predictions = model.predict(X)
    try:
        probabilities = model.predict_proba(X)
    except:
        probabilities = None
    
    return predictions, probabilities


def predict_ann(model, features: torch.Tensor):
    """Make predictions with ANN model"""
    if features.ndim == 1:
        features = features.unsqueeze(0)
    
    with torch.no_grad():
        outputs = model(features)
        probabilities = F.softmax(outputs, dim=1).numpy()
        predictions = outputs.argmax(dim=1).numpy()
    
    return predictions, probabilities



# ============================================
# MAIN INFERENCE FUNCTION
# ============================================
def run_inference():
    """Main function to run inference on PAIP data using TCGA trained models (fold3 only)"""
    
    # Load ground truth
    print("="*80)
    print("Loading Ground Truth Labels")
    print("="*80)
    ground_truth_df = load_ground_truth(GROUND_TRUTH_PATH)
    
    # Results storage
    all_results = []
    
    # Iterate through each feature model (Conch, H-Optimus, UNI)
    for feature_model in FEATURE_MODELS:
        print(f"\n{'='*80}")
        print(f"Processing Feature Model: {feature_model}")
        print(f"{'='*80}")
        
        # Load PAIP features
        paip_feature_dir = os.path.join(PAIP_DATA_BASE, PAIP_FEATURE_DIRS[feature_model])
        features_dict, wsi_ids = load_paip_features(paip_feature_dir)
        
        if not features_dict:
            print(f"No features found for {feature_model}, skipping...")
            continue
        
        # Get input dimension
        sample_feature = next(iter(features_dict.values()))
        input_dim = sample_feature.shape[0]
        print(f"Feature dimension: {input_dim}")
        
        # Iterate through each classifier type
        for classifier_name, model_filename in CLASSIFIER_TYPES.items():
            print(f"\n{'-'*60}")
            print(f"Classifier: {classifier_name}")
            print(f"{'-'*60}")
            
            # Model path for fold3
            model_path = os.path.join(
                TCGA_MODELS_BASE,
                feature_model,
                TASK,
                'models',
                model_filename
            )
            
            if not os.path.exists(model_path):
                print(f"Model not found: {model_path}")
                continue
            
            print(f"Loading model: {model_filename}")
            
            # Load model
            try:
                if classifier_name == 'ann':
                    model = load_ann_model(model_path, input_dim)
                else:
                    model = load_sklearn_model(model_path)
            except Exception as e:
                print(f"Error loading model: {e}")
                continue
            
            # Make predictions on all PAIP samples
            predictions = []
            probabilities = []
            true_labels = []
            valid_wsi_ids = []
            
            for wsi_id in wsi_ids:
                # Get features
                features = features_dict[wsi_id]
                
                # Get true label
                true_label = get_label_from_wsi_id(wsi_id, ground_truth_df)
                if true_label is None:
                    continue
                
                # Make prediction
                try:
                    if classifier_name == 'ann':
                        pred, proba = predict_ann(model, features)
                    else:
                        pred, proba = predict_sklearn(model, features)
                    
                    predictions.append(pred[0] if isinstance(pred, np.ndarray) else pred)
                    probabilities.append(proba[0] if proba is not None else None)
                    true_labels.append(true_label)
                    valid_wsi_ids.append(wsi_id)
                except Exception as e:
                    print(f"Error predicting for {wsi_id}: {e}")
                    continue
            
            print(f"Processed {len(predictions)} samples")
            
            # Calculate metrics
            if predictions:
                probabilities_array = np.array(probabilities) if probabilities[0] is not None else None
                
                metrics = calculate_metrics(
                    np.array(true_labels),
                    np.array(predictions),
                    probabilities_array
                )
                
                print(f"\nResults for {feature_model} - {classifier_name}:")
                print(f"  Accuracy: {metrics['accuracy']:.4f}")
                print(f"  Balanced Accuracy: {metrics['balanced_accuracy']:.4f}")
                print(f"  F1 Macro: {metrics['f1_macro']:.4f}")
                print(f"  F1 Weighted: {metrics['f1_weighted']:.4f}")
                if metrics.get('auroc'):
                    print(f"  AUROC: {metrics['auroc']:.4f}")
                print(f"  Confusion Matrix: {metrics['confusion_matrix']}")
                
                # Store results
                result_entry = {
                    'feature_model': feature_model,
                    'classifier': classifier_name,
                    'fold': FOLD_TO_USE,
                    'num_samples': len(true_labels),
                    **metrics
                }
                all_results.append(result_entry)
    
    # Save results to CSV
    results_df = pd.DataFrame(all_results)
    output_file = os.path.join(OUTPUT_PATH, f"tcga_to_paip_inference_{TASK}_fold{FOLD_TO_USE}.csv")
    results_df.to_csv(output_file, index=False)
    print(f"\n{'='*80}")
    print(f"Results saved to: {output_file}")
    print(f"{'='*80}")
    
    return results_df


# ============================================
# RUN INFERENCE
# ============================================
if __name__ == "__main__":
    results = run_inference()
    print("\n" + "="*80)
    print("INFERENCE COMPLETE!")
    print("="*80)
    print(results.to_string())


In [ ]:
# Run the inference
results = run_inference()

# Display results
print("\n" + "="*80)
print("FINAL RESULTS SUMMARY")
print("="*80)
print(results.to_string())

=== CELL 9: Visualize Results (Optional) ===
import matplotlib.pyplot as plt
import seaborn as sns

# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Plot 1: Accuracy comparison
ax1 = axes[0, 0]
pivot_acc = results.pivot(index='classifier', columns='feature_model', values='accuracy')
pivot_acc.plot(kind='bar', ax=ax1)
ax1.set_title('Accuracy by Classifier and Feature Model')
ax1.set_ylabel('Accuracy')
ax1.legend(title='Feature Model')
ax1.grid(True, alpha=0.3)

# Plot 2: Balanced Accuracy comparison
ax2 = axes[0, 1]
pivot_bacc = results.pivot(index='classifier', columns='feature_model', values='balanced_accuracy')
pivot_bacc.plot(kind='bar', ax=ax2)
ax2.set_title('Balanced Accuracy by Classifier and Feature Model')
ax2.set_ylabel('Balanced Accuracy')
ax2.legend(title='Feature Model')
ax2.grid(True, alpha=0.3)

# Plot 3: F1 Macro comparison
ax3 = axes[1, 0]
pivot_f1 = results.pivot(index='classifier', columns='feature_model', values='f1_macro')
pivot_f1.plot(kind='bar', ax=ax3)
ax3.set_title('F1 Macro by Classifier and Feature Model')
ax3.set_ylabel('F1 Macro')
ax3.legend(title='Feature Model')
ax3.grid(True, alpha=0.3)

# Plot 4: AUROC comparison
ax4 = axes[1, 1]
if 'auroc' in results.columns:
    pivot_auroc = results.pivot(index='classifier', columns='feature_model', values='auroc')
    pivot_auroc.plot(kind='bar', ax=ax4)
    ax4.set_title('AUROC by Classifier and Feature Model')
    ax4.set_ylabel('AUROC')
    ax4.legend(title='Feature Model')
    ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, f'inference_results_{TASK}.png'), dpi=300, bbox_inches='tight')
plt.show()

print("✅ Visualization complete!")

In [3]:
"""
Test Script for TCGA to PAIP Inference
This script tests if all paths, files, and configurations are correct before running full inference
"""

import os
import torch
import pandas as pd
import joblib

# ============================================
# CONFIGURATION (same as main script)
# ============================================
TCGA_MODELS_BASE = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\slide_classification\TCGA_Result\5-Caption_based_aggregation"
PAIP_DATA_BASE = r"D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\Caption_Based_Clustering_FiveCrop"
GROUND_TRUTH_PATH = r"D:\Aamir Gulzar\KSA_project2\paip_data\labels\paip_78_labels.csv"

FEATURE_MODELS = ['Conch1_5', 'H-Optimus-1', 'UNI2']
PAIP_FEATURE_DIRS = {
    'Conch1_5': 'conch_CC_fivecrop',
    'H-Optimus-1': 'hoptimus_CC_fivecrop',
    'UNI2': 'uni2_CC_fivecrop'
}
CLASSIFIER_TYPES = {
    'logistic_regression': 'fold3_logistic_regression.pkl',
    'ann': 'fold3_trained_ann_model_10752.pth',
    'knn': 'fold3_knn_model.pkl',
    'protonet': 'fold3_protonet_model.pkl'
}
TASK = 'MSIH'

print("="*80)
print("TCGA to PAIP Inference - Setup Test")
print("="*80)

# ============================================
# TEST 1: Check Base Directories
# ============================================
print("\n[TEST 1] Checking base directories...")
test_passed = True

if os.path.exists(TCGA_MODELS_BASE):
    print(f"✅ TCGA models base directory exists")
else:
    print(f"❌ TCGA models base directory NOT FOUND: {TCGA_MODELS_BASE}")
    test_passed = False

if os.path.exists(PAIP_DATA_BASE):
    print(f"✅ PAIP data base directory exists")
else:
    print(f"❌ PAIP data base directory NOT FOUND: {PAIP_DATA_BASE}")
    test_passed = False

if os.path.exists(GROUND_TRUTH_PATH):
    print(f"✅ Ground truth CSV exists")
else:
    print(f"❌ Ground truth CSV NOT FOUND: {GROUND_TRUTH_PATH}")
    test_passed = False

# ============================================
# TEST 2: Check Ground Truth CSV Structure
# ============================================
print("\n[TEST 2] Checking ground truth CSV structure...")
try:
    df = pd.read_csv(GROUND_TRUTH_PATH)
    print(f"✅ Ground truth CSV loaded successfully")
    print(f"   - Total samples: {len(df)}")
    print(f"   - Columns: {df.columns.tolist()}")
    
    # Check for required columns
    if 'WSI_Id' in df.columns:
        print(f"✅ 'WSI_Id' column found")
        print(f"   - Sample WSI_Ids: {df['WSI_Id'].head(3).tolist()}")
    else:
        print(f"❌ 'WSI_Id' column NOT FOUND")
        test_passed = False
    
    # Check for label columns
    if 'label' in df.columns:
        print(f"✅ 'label' column found")
        print(f"   - Unique labels: {df['label'].unique()}")
    elif 'label_desc' in df.columns:
        print(f"✅ 'label_desc' column found")
        print(f"   - Unique labels: {df['label_desc'].unique()}")
    else:
        print(f"❌ No label column found (need 'label' or 'label_desc')")
        test_passed = False
        
except Exception as e:
    print(f"❌ Error loading ground truth CSV: {e}")
    test_passed = False

# ============================================
# TEST 3: Check TCGA Model Files
# ============================================
print("\n[TEST 3] Checking TCGA model files...")
models_found = 0
models_missing = 0

for feature_model in FEATURE_MODELS:
    print(f"\n  Feature Model: {feature_model}")
    model_dir = os.path.join(TCGA_MODELS_BASE, feature_model, TASK, 'models')
    
    if not os.path.exists(model_dir):
        print(f"  ❌ Model directory NOT FOUND: {model_dir}")
        models_missing += len(CLASSIFIER_TYPES)
        continue
    
    for classifier_name, model_filename in CLASSIFIER_TYPES.items():
        model_path = os.path.join(model_dir, model_filename)
        if os.path.exists(model_path):
            file_size = os.path.getsize(model_path) / 1024  # KB
            print(f"  ✅ {classifier_name}: {model_filename} ({file_size:.1f} KB)")
            models_found += 1
        else:
            print(f"  ❌ {classifier_name}: {model_filename} NOT FOUND")
            models_missing += 1

print(f"\n  Summary: {models_found} models found, {models_missing} missing")
if models_missing > 0:
    test_passed = False

# ============================================
# TEST 4: Check PAIP Feature Files
# ============================================
print("\n[TEST 4] Checking PAIP feature files...")
features_found = 0

for feature_model in FEATURE_MODELS:
    feature_dir_name = PAIP_FEATURE_DIRS[feature_model]
    feature_dir = os.path.join(PAIP_DATA_BASE, feature_dir_name)
    
    print(f"\n  Feature Model: {feature_model}")
    print(f"  Directory: {feature_dir_name}")
    
    if not os.path.exists(feature_dir):
        print(f"  ❌ Feature directory NOT FOUND")
        test_passed = False
        continue
    
    # Count .pt files
    pt_files = [f for f in os.listdir(feature_dir) if f.endswith('.pt')]
    print(f"  ✅ Found {len(pt_files)} .pt files")
    
    if len(pt_files) > 0:
        features_found += len(pt_files)
        # Show first 3 files
        print(f"  Sample files: {pt_files[:3]}")
        
        # Test loading one file
        try:
            test_file = os.path.join(feature_dir, pt_files[0])
            test_features = torch.load(test_file)
            if test_features.dim() > 1:
                test_features = test_features.flatten()
            print(f"  ✅ Test load successful - Feature dimension: {test_features.shape[0]}")
        except Exception as e:
            print(f"  ❌ Error loading test file: {e}")
            test_passed = False
    else:
        print(f"  ❌ No .pt files found")
        test_passed = False

print(f"\n  Total feature files found: {features_found}")

# ============================================
# TEST 5: Test Model Loading
# ============================================
print("\n[TEST 5] Testing model loading...")

# Test loading one model from each type
test_feature_model = 'Conch1_5'
model_dir = os.path.join(TCGA_MODELS_BASE, test_feature_model, TASK, 'models')

if os.path.exists(model_dir):
    print(f"  Testing with {test_feature_model} models...")
    
    # Test sklearn model (logistic regression)
    lr_path = os.path.join(model_dir, CLASSIFIER_TYPES['logistic_regression'])
    if os.path.exists(lr_path):
        try:
            lr_model = joblib.load(lr_path)
            print(f"  ✅ Logistic Regression model loaded successfully")
        except Exception as e:
            print(f"  ❌ Error loading Logistic Regression: {e}")
            test_passed = False
    
    # Test ANN model
    ann_path = os.path.join(model_dir, CLASSIFIER_TYPES['ann'])
    if os.path.exists(ann_path):
        try:
            ann_state = torch.load(ann_path, map_location='cpu')
            print(f"  ✅ ANN model state dict loaded successfully")
            print(f"     Keys: {list(ann_state.keys())[:3]}...")
        except Exception as e:
            print(f"  ❌ Error loading ANN: {e}")
            test_passed = False
else:
    print(f"  ⚠️  Skipping model loading test - directory not found")

# ============================================
# TEST 6: Check Feature-Label Matching
# ============================================
print("\n[TEST 6] Checking feature-label matching...")

try:
    df = pd.read_csv(GROUND_TRUTH_PATH)
    ground_truth_ids = set(df['WSI_Id'].tolist())
    
    # Check first feature directory
    test_feature_dir = os.path.join(PAIP_DATA_BASE, PAIP_FEATURE_DIRS['Conch1_5'])
    if os.path.exists(test_feature_dir):
        pt_files = [f for f in os.listdir(test_feature_dir) if f.endswith('.pt')]
        feature_ids = set([os.path.splitext(f)[0] for f in pt_files])
        
        # Find matches
        matching_ids = ground_truth_ids.intersection(feature_ids)
        print(f"  Ground truth samples: {len(ground_truth_ids)}")
        print(f"  Feature files: {len(feature_ids)}")
        print(f"  ✅ Matching samples: {len(matching_ids)}")
        
        if len(matching_ids) == 0:
            print(f"  ❌ WARNING: No matching samples found!")
            print(f"  Sample ground truth IDs: {list(ground_truth_ids)[:3]}")
            print(f"  Sample feature IDs: {list(feature_ids)[:3]}")
            test_passed = False
        elif len(matching_ids) < 10:
            print(f"  ⚠️  WARNING: Very few matching samples ({len(matching_ids)})")
    
except Exception as e:
    print(f"  ❌ Error checking matching: {e}")
    test_passed = False

# ============================================
# FINAL SUMMARY
# ============================================
print("\n" + "="*80)
if test_passed:
    print("✅ ALL TESTS PASSED - Ready to run inference!")
    print("="*80)
    print("\nNext step: Run the main inference script")
    print("  python tcga_to_paip_inference.py")
else:
    print("❌ SOME TESTS FAILED - Please fix issues before running inference")
    print("="*80)
    print("\nPlease review the errors above and:")
    print("  1. Check that all paths are correct")
    print("  2. Verify model files exist")
    print("  3. Ensure feature files are in the right location")
    print("  4. Check that WSI_Ids match between features and ground truth")

print("\n" + "="*80)


TCGA to PAIP Inference - Setup Test

[TEST 1] Checking base directories...
✅ TCGA models base directory exists
✅ PAIP data base directory exists
✅ Ground truth CSV exists

[TEST 2] Checking ground truth CSV structure...
✅ Ground truth CSV loaded successfully
   - Total samples: 78
   - Columns: ['WSI_Id', 'label']
✅ 'WSI_Id' column found
   - Sample WSI_Ids: ['training_data_01_MSIH', 'training_data_02_nonMSIH', 'training_data_03_nonMSIH']
✅ 'label' column found
   - Unique labels: ['MSIH' 'nonMSIH']

[TEST 3] Checking TCGA model files...

  Feature Model: Conch1_5
  ❌ logistic_regression: fold3_logistic_regression.pkl NOT FOUND
  ❌ ann: fold3_trained_ann_model_10752.pth NOT FOUND
  ❌ knn: fold3_knn_model.pkl NOT FOUND
  ❌ protonet: fold3_protonet_model.pkl NOT FOUND

  Feature Model: H-Optimus-1
  ✅ logistic_regression: fold3_logistic_regression.pkl (168.8 KB)
  ❌ ann: fold3_trained_ann_model_10752.pth NOT FOUND
  ✅ knn: fold3_knn_model.pkl (26211.2 KB)
  ✅ protonet: fold3_protonet_mod